In [1]:
from action import take_action
from screenshot import get_screen_with_grid

import requests
import re

import cv2

In [2]:
url = "http://127.0.0.1:8000/predict/"


In [3]:

def send_image():
    images = get_screen_with_grid()

    payload = images[1]

    # 1. Encode the numpy array to an image format (like PNG)
    success, encoded_image = cv2.imencode('.png', payload)
    image_bytes = encoded_image.tobytes()

    response = requests.post(url, files={"file": ("screenshot.png", image_bytes, "image/png")})

    return response

In [17]:
import json

def get_actions(ai_response):
    # Depending on how the API returns data, response.json() might be a string containing JSON
    try:
        # First parsing: get the string out of the requests response
        response_data = ai_response.json()
        
        # If response_data is a string (double-encoded JSON), we parse it again
        if isinstance(response_data, str):
            return [json.loads(response_data)]
        else:
            return [response_data]
            
    except json.JSONDecodeError:
        # Fallback if it's not strictly JSON, try ast.literal_eval or string manipulation
        import ast
        return [ast.literal_eval(ai_response.text.strip('"').replace('\\"', '"'))]

In [6]:
response = send_image()

<Response [200]>

In [7]:
response.text

'"{\\"action\\": \\"click\\", \\"target\\": \\"AL1\\"}"'

In [18]:
actions = get_actions(response)
actions

[{'action': 'click', 'target': 'AL1'}]

In [5]:
def take_and_encode_screenshot():
    """Takes screenshot, overlays grid, and returns JPEG bytes."""
    images = get_screen_with_grid()
    payload = images[1] # The grid overlay Image array
    
    # Needs to be converted to RGB for JPEG encoding, though openCV imencode handles BGR to JPEG natively if we use cv2
    success, encoded_image = cv2.imencode('.jpg', payload)
    if success:
        return payload, encoded_image.tobytes()
    return None, None

payload_image, image_bytes = take_and_encode_screenshot()

# Display what the model is seeing using OpenCV
cv2.imshow("What the Model Sees", payload_image)
cv2.waitKey(0) # Press any key to close the window
cv2.destroyAllWindows()